# 150 — Attention Distance Analysis

How far does each attention head reach? Some heads attend locally (nearby tokens),
while others attend globally (distant tokens). This module provides tools to
measure attention distance, classify heads, and trace information flow.

## Why This Matters

Attention distance reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_distance import (
    mean_attention_distance,
    local_vs_global_heads,
    distance_weighted_flow,
    attention_decay_curve,
    receptive_field,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)

# Random init
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.arange(20)

## Mean Attention Distance

For each head, compute the expected distance between query and key positions,
weighted by attention probability. Higher values mean the head attends further.

In [ ]:
result = mean_attention_distance(model, tokens)

print(f"Overall mean distance: {result['overall_mean_distance']:.2f} positions\n")
for h in result['per_head']:
    bar = '#' * int(h['mean_distance'] * 3)
    print(f"L{h['layer']}H{h['head']}: mean={h['mean_distance']:.2f}  max={h['max_distance']:.2f}  {bar}")

## Local vs Global Heads

Classify each head as local (most attention within a window), global (most attention
outside the window), or mixed.

In [ ]:
result = local_vs_global_heads(model, tokens, local_window=3)

print(f"Local: {result['n_local']}, Global: {result['n_global']}, Mixed: {result['n_mixed']}\n")
for h in result['per_head']:
    tag = h['classification'].upper()
    print(f"L{h['layer']}H{h['head']}: [{tag:6s}] local_mass={h['local_attention_mass']:.3f}")

## Distance-Weighted Flow

Track how information from a specific source position spreads through the network.
Shows how much attention later positions pay to the source, broken down by distance.

In [ ]:
result = distance_weighted_flow(model, tokens, source_pos=0)

for layer_info in result['per_layer']:
    print(f"Layer {layer_info['layer']}: total attention received = {layer_info['total_attention_received']:.4f}")
    for f in layer_info['flow_by_distance'][:5]:
        bar = '#' * int(f['attention_to_source'] * 100)
        print(f"  dist={f['distance']:2d}: {f['attention_to_source']:.4f} {bar}")
    print()

## Attention Decay Curve

For a specific query position, measure how attention decays as a function of
distance to the key. The half-life tells you how far the head "remembers".

In [ ]:
result = attention_decay_curve(model, tokens, query_pos=-1)

print(f"Query position: {result['query_position']}\n")
for h in result['per_head']:
    hl = h['half_life']
    hl_str = f"{hl}" if hl >= 0 else "N/A"
    print(f"L{h['layer']}H{h['head']}: max_attn={h['max_attention']:.4f}  half_life={hl_str}")

## Receptive Field

Estimate which input positions have significant influence on a target position,
layer by layer. A wider receptive field means the position integrates more context.

In [ ]:
result = receptive_field(model, tokens, target_pos=-1, threshold=0.05)

print(f"Target: position {result['target_position']}")
print(f"Max receptive field width: {result['max_field_width']}\n")

for layer_info in result['per_layer']:
    positions = layer_info['positions_in_field']
    print(f"Layer {layer_info['layer']}: width={layer_info['field_width']}  "
          f"range=[{layer_info['field_start']}-{layer_info['field_end']}]  "
          f"positions={positions}")

## Comparing Thresholds

Lower thresholds include more positions in the receptive field.

In [ ]:
for thresh in [0.01, 0.05, 0.1, 0.2]:
    r = receptive_field(model, tokens, threshold=thresh)
    print(f"threshold={thresh:.2f}: max_field_width={r['max_field_width']}")